# 00 — Source Recovery and Full-Text Status

Purpose: turn the identified source leads into a transparent source-recovery ledger. The notebook checks what is already local, tries safe public retrieval where possible, validates whether a file is a real PDF/HTML/text source, extracts whatever text can be extracted, and writes a manual RA queue for anything blocked, paywalled, or still missing.

This notebook does **not** decide inclusion for the paper. It only answers: what source text do we actually have, and what still needs manual download?


In [ ]:
# Optional first-time setup:
# %pip install -q pandas requests beautifulsoup4 pdfplumber pypdf

from pathlib import Path
import os, re, csv, json, shutil, hashlib, time
from urllib.parse import urlparse

import pandas as pd
import requests

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None

# Walk upward until the DC_job_potential folder is found.
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "Media_Article").is_dir() and (p / "Research_Workflow").is_dir():
            return p
    raise RuntimeError("Could not locate DC_job_potential root. Run this notebook inside the project folder.")

ROOT = find_project_root(Path.cwd())
WORKFLOW = ROOT / "Research_Workflow"
EXTRACTION = WORKFLOW / "02_Source_Extraction"
REGISTER_DIR = EXTRACTION / "data" / "source_register"
DATA_DIR = EXTRACTION / "data"
RAW_DIR = DATA_DIR / "raw_sources"
TEXT_DIR = DATA_DIR / "fulltext"
QUEUE_DIR = DATA_DIR / "manual_RA_queue"
OUT_DIR = DATA_DIR / "coding_outputs"
LEGACY_SOURCES = ROOT / "Media_Article" / "Sources"

for d in (RAW_DIR, TEXT_DIR, QUEUE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SOURCE_REGISTER = REGISTER_DIR / "source_leads_seed.csv"
print("ROOT:", ROOT)
print("SOURCE_REGISTER:", SOURCE_REGISTER)
print("LEGACY_SOURCES:", LEGACY_SOURCES)


In [ ]:
# Controls
# Leave True to try public, non-authenticated downloads. Set False for a local-only audit.
RUN_DOWNLOADS = True  # public/direct downloads enabled; manual/paywalled sources remain queued

# Keep browser/manual-only sources out of automated download unless a direct PDF URL is present.
SKIP_MANUAL_ACCESS_MODES = {"paywall_or_library", "manual_or_browser_download"}

REQUEST_TIMEOUT = 35
USER_AGENT = "Mozilla/5.0 (compatible; DCJobPotentialSourceRecovery/1.0; academic source audit)"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "text/html,application/pdf,*/*"}

leads = pd.read_csv(SOURCE_REGISTER)
leads["priority"] = pd.to_numeric(leads["priority"], errors="coerce").fillna(9).astype(int)
leads = leads.sort_values(["priority", "source_id"]).reset_index(drop=True)
leads.head(20)


In [ ]:
def safe_filename(name: str, default="source") -> str:
    name = str(name or default)
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("_")
    return name[:180] or default

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def classify_bytes(data: bytes, url="") -> dict:
    head = data[:4096]
    text_head = head.decode("utf-8", errors="ignore").lower()
    if data.startswith(b"%PDF-"):
        return {"content_class": "pdf", "is_extractable": True, "block_reason": ""}
    if "just a moment" in text_head or "cloudflare" in text_head or "enable javascript" in text_head:
        return {"content_class": "html_challenge", "is_extractable": False, "block_reason": "javascript_or_cloudflare_challenge"}
    if "ssrn" in (url or "").lower() and ("download" not in (url or "").lower()) and b"%PDF-" not in head:
        return {"content_class": "html_landing", "is_extractable": True, "block_reason": "landing_page_not_pdf"}
    if b"<html" in head.lower() or b"<!doctype html" in head.lower():
        return {"content_class": "html", "is_extractable": True, "block_reason": ""}
    if len(data) == 0:
        return {"content_class": "empty", "is_extractable": False, "block_reason": "empty_file"}
    return {"content_class": "unknown_binary_or_text", "is_extractable": False, "block_reason": "unknown_format"}

def classify_path(path: Path, url="") -> dict:
    if not path.exists():
        return {"content_class": "missing", "is_extractable": False, "block_reason": "file_missing"}
    data = path.read_bytes()
    d = classify_bytes(data, url=url)
    d.update({"bytes": len(data), "sha256": sha256_bytes(data) if data else ""})
    return d

def download_one(url: str) -> tuple[bytes, str, str]:
    r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    ctype = r.headers.get("content-type", "")
    final_url = r.url
    r.raise_for_status()
    return r.content, ctype, final_url


In [ ]:
# Audit local legacy files and recover/download source leads where possible.
rows = []

for _, lead in leads.iterrows():
    sid = lead["source_id"]
    expected = safe_filename(lead.get("expected_file_name") or f"{sid}.pdf")
    raw_path = RAW_DIR / expected
    legacy_path = LEGACY_SOURCES / expected
    status = {
        "source_id": sid,
        "asset_type": lead.get("asset_type", ""),
        "priority": lead.get("priority", ""),
        "title": lead.get("title", ""),
        "url": lead.get("url", ""),
        "access_mode": lead.get("access_mode", ""),
        "expected_file_name": expected,
        "raw_path": str(raw_path),
        "legacy_path": str(legacy_path) if legacy_path.exists() else "",
        "retrieval_status": "not_attempted",
        "content_type_header": "",
        "final_url": "",
        "content_class": "missing",
        "is_extractable": False,
        "block_reason": "",
        "notes": lead.get("notes", ""),
    }

    # 1) Prefer copied/recovered raw source, then legacy file.
    if raw_path.exists():
        status.update(classify_path(raw_path, url=lead.get("url", "")))
        status["retrieval_status"] = "already_in_raw_sources"
    elif legacy_path.exists():
        raw_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(legacy_path, raw_path)
        status.update(classify_path(raw_path, url=lead.get("url", "")))
        status["retrieval_status"] = "copied_from_legacy_sources"
    else:
        access_mode = str(lead.get("access_mode", ""))
        url = str(lead.get("url", "")).strip()
        direct_url_pdf = url.lower().split("?", 1)[0].endswith(".pdf")
        should_try = RUN_DOWNLOADS and bool(url) and (access_mode not in SKIP_MANUAL_ACCESS_MODES or direct_url_pdf)
        if raw_path.exists() and status.get("content_class") == "html_challenge":
            # Do not overwrite known stubs/challenge pages automatically. Leave them for RA/manual browser recovery.
            status["retrieval_status"] = "manual_required_existing_stub"
            status["block_reason"] = status.get("block_reason") or "existing_html_challenge_stub"
        elif should_try:
            try:
                data, ctype, final_url = download_one(url)
                raw_path.write_bytes(data)
                status["content_type_header"] = ctype
                status["final_url"] = final_url
                status.update(classify_path(raw_path, url=final_url))
                status["retrieval_status"] = "downloaded"
            except Exception as e:
                status["retrieval_status"] = "download_failed"
                status["block_reason"] = f"{type(e).__name__}: {e}"
        else:
            status["retrieval_status"] = "manual_required_or_download_disabled"
            if access_mode in SKIP_MANUAL_ACCESS_MODES:
                status["block_reason"] = access_mode
            elif not RUN_DOWNLOADS:
                status["block_reason"] = "RUN_DOWNLOADS_false"
            else:
                status["block_reason"] = "no_url"

    rows.append(status)

status_df = pd.DataFrame(rows)
status_path = OUT_DIR / "source_recovery_status.csv"
status_df.to_csv(status_path, index=False)
print("Wrote", status_path)
status_df[["source_id", "priority", "retrieval_status", "content_class", "is_extractable", "block_reason"]].head(30)


In [ ]:
# Extract text from whatever is extractable.
def extract_pdf_text(path: Path) -> str:
    pages = []
    try:
        import pdfplumber
        with pdfplumber.open(str(path)) as pdf:
            for i, pg in enumerate(pdf.pages, 1):
                pages.append(f"[[PAGE {i}]]\n" + (pg.extract_text() or ""))
        return "\n\n".join(pages)
    except Exception as e1:
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            for i, pg in enumerate(reader.pages, 1):
                pages.append(f"[[PAGE {i}]]\n" + (pg.extract_text() or ""))
            return "\n\n".join(pages)
        except Exception as e2:
            return f"[[EXTRACTION_ERROR]] pdfplumber={e1}; pypdf={e2}"

def html_to_text(path: Path) -> str:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if BeautifulSoup is None:
        return re.sub(r"<[^>]+>", " ", raw)
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return soup.get_text("\n", strip=True)

text_rows = []
for _, r in status_df.iterrows():
    raw_path = Path(r["raw_path"])
    if not bool(r["is_extractable"]) or not raw_path.exists():
        continue
    cls = r["content_class"]
    if cls == "pdf":
        text = extract_pdf_text(raw_path)
    elif cls in {"html", "html_landing"}:
        text = html_to_text(raw_path)
    else:
        continue
    out = TEXT_DIR / f"{safe_filename(r['source_id'])}.txt"
    out.write_text(text, encoding="utf-8")
    words = len(re.findall(r"\w+", text))
    text_rows.append({
        "source_id": r["source_id"], "text_file": str(out), "content_class": cls,
        "words": words, "source_raw_path": str(raw_path), "usable_for_extraction": words >= 250
    })

text_df = pd.DataFrame(text_rows)
text_inv = OUT_DIR / "fulltext_inventory.csv"
text_df.to_csv(text_inv, index=False)
print("Wrote", text_inv)
text_df.sort_values(["usable_for_extraction", "words"], ascending=[True, False]).head(40) if not text_df.empty else "No text extracted"


In [ ]:
# Build the RA manual-download queue.
if text_rows:
    usable = set(text_df.loc[text_df["usable_for_extraction"], "source_id"])
else:
    usable = set()

queue = status_df[(~status_df["source_id"].isin(usable)) | (status_df["content_class"].isin(["html_challenge", "missing"]))].copy()
queue["ra_instruction"] = queue.apply(
    lambda r: (
        "Download real PDF/full text manually; current file is an HTML challenge/stub."
        if r["content_class"] == "html_challenge" else
        "Find official PDF/full text manually, save using expected_file_name, then rerun this notebook."
        if r["content_class"] == "missing" else
        "Check whether the web page contains a PDF link or enough text for extraction."
    ), axis=1
)
queue = queue[[
    "priority", "source_id", "title", "url", "expected_file_name", "retrieval_status",
    "content_class", "block_reason", "ra_instruction", "notes"
]].sort_values(["priority", "source_id"])
queue_path = QUEUE_DIR / "manual_download_queue.csv"
queue.to_csv(queue_path, index=False)
print("Wrote", queue_path)
queue.head(50)


## Handoff Rule

After RA manual downloads, save each file using `expected_file_name` into `Research_Workflow/02_Source_Extraction/data/raw_sources/`, then rerun this notebook. The status, text inventory, and queue will update without changing the source register.
